## Setup

In [ ]:
# Imports

import polars as pl
import polars.selectors as cs
import pandas as pd

from glob import glob
from pathlib import Path


In [ ]:
# GLOBAL VARIABLES

COUNTIES = ['REEVES', 'LOVING', 'WARD', 'PECOS', 'JEFF DAVIS', 'CULBERSON']

# Create project root
def find_project_root(marker: str = ".git") -> Path:
    """Climbs up folders starting from CWD until it finds the root."""
    current = Path.cwd().resolve()
    
    # Loop upward until hitting the project root (where current == its own parent)
    while current != current.parent:
        if (current / marker).exists():
            return current
        current = current.parent

PROJECT_ROOT = find_project_root(marker=".git")

## Fetch Data

## Convert to csv

In [ ]:
# Convert to csv

def html_to_csv(file_path: str):
    # read_html returns two tables. First table is useless, (1,1) shape. Second table contains the desired data.
    tables = pd.read_html(file_path)
    df = tables[1]

    # Create new path
    file_path = Path(file_path)
    new_path = file_path.with_suffix('.csv')

    # Replace old file
    df.to_csv(new_path, index=False)
    file_path.unlink()

# Folders with html files in question
html_folder_paths = ['air-permits']

# Aggregate all html files together
all_html_file_paths = []
for html_folder_path in html_folder_paths:
    html_file_paths = glob(str(PROJECT_ROOT / 'data' / 'raw' / html_folder_path / '**' / '*.xls'))

    all_html_file_paths.extend(html_file_paths)

# Convert all html files
for html_file_path in all_html_file_paths:
    html_to_csv(html_file_path)


## Filter

### Permit Query Filtering

In [ ]:
# Filter out relevant permits to create candidate list

def permit_filter(folder_path: str, relevant_cols: list[str], exact_filters: dict = None) -> pl.DataFrame:
    df = pl.read_csv(folder_path)

    # Check for exact matches
    if exact_filters is not None:
        df = df.filter(
            *[
                pl.col(col) == content
                for col, content in exact_filters.items()
            ]
        )

    # Get relevant counties
    df = df.filter(
        pl.col('County Name').is_in(COUNTIES)
    )

    # Casting
    df = df.with_columns(
        pl.col('TCEQ Received Date').str.to_datetime(format="%m/%d/%Y")
    )

    # Group by permit number to remove dupes. Select the latest permit entry as it reflects recent status
    df = (
        df
        .group_by('Permit Number')
        .agg(
            pl.all()
            .sort_by('TCEQ Received Date')
            .last()
        )
    )

    # Get relevant columns
    df = df.select(relevant_cols)

    return df

relevant_cols = ['Permit Number', 'Permit Status', 'TCEQ Received Date', 'Project Number', 'Legal Name', 'Project Name', 'County Name']

# Create a dataframe for each permit type
std_folder_path = str(PROJECT_ROOT / 'data' / 'raw' / 'air-permits' / 'std' / '*.csv')
std_df = permit_filter(std_folder_path, relevant_cols, {'Rules': '6005'})

psd_folder_path = str(PROJECT_ROOT / 'data' / 'raw' / 'air-permits' / 'psd' / '*.csv')
psd_df = permit_filter(psd_folder_path, relevant_cols)

ghgpsd_folder_path = str(PROJECT_ROOT / 'data' / 'raw' / 'air-permits' / 'ghgpsd' / '*.csv')
ghgpsd_df = permit_filter(ghgpsd_folder_path, relevant_cols)

# Combine them and add relevant columns
btm_df = pl.concat([std_df, psd_df, ghgpsd_df])
btm_df = (
    btm_df
    .group_by('Project Number')
    .agg(
        pl.exclude('Project Number').last()
    )
)

btm_df = btm_df.with_columns(
    pl.lit(None, dtype=pl.Utf8).alias('BTM Onsite Datacenter'),
    pl.lit(None, dtype=pl.Utf8).alias('Site Name'),
    pl.lit(None, dtype=pl.Int32).alias('Capacity (MW)'),
    pl.lit(None, dtype=pl.Utf8).alias('Turbine / Engine Model'),
    pl.lit(None, dtype=pl.Utf8).alias('OEM'),
    pl.lit(None, dtype=pl.Utf8).alias('Status'),
    pl.lit(None, dtype=pl.Utf8).alias('Sources'),
)

# Export so rest can be done manually
candidates_file_path = PROJECT_ROOT / 'data' / 'processed' / 'candidates-list.csv'

# Uncomment to create csv
# btm_df.write_csv(candidates_file_path)

### Filter to Final Table

In [ ]:
# Selecting relevant information from candidates table

def get_final_table(file_path: str, final_cols: list[str]):
    df = pl.read_csv(file_path)

    df = (
        df

        # Remove facilities that are not BTM gas (onsite) data centers
        .filter(
            pl.col('BTM Onsite Datacenter') == True
        )
        # Get final columns
        .select(final_cols)
    )

    return df

final_cols = ['Site Name', 'County Name', 'Capacity (MW)', 'Turbine / Engine Model', 'OEM', 'Status', 'Sources']
data_centers_df = get_final_table(candidates_file_path, final_cols)

In [ ]:
# Adding manual entries and exporting

manual_entries = pl.DataFrame(
    {
        'Site Name': ['Alpha Digital Campus'],
        'County Name': ['Reeves'],
        'Capacity (MW)': [2000.0],
        'Turbine / Engine Model': [None],
        'OEM': [None],
        'Status': ['Announced'],
        'Sources': ['LandBridge Press Release']

    }
)

data_centers_df = data_centers_df.vstack(manual_entries)

# Uncomment to create csv
# data_center_file_path = PROJECT_ROOT / 'data' / 'processed' / 'data-center-list.csv'
# data_centers_df.write_csv(data_center_file_path)

In [28]:
data_centers_df

Site Name,County Name,Capacity (MW),Turbine / Engine Model,OEM,Status,Sources
str,str,f64,str,str,str,str
"""Circe Energy Data Centers E Mo…","""WARD""",400.0,"""['HSK78G']""","""['Cummins']""","""Permitted""","""['TCEQ NSR Air Permit', 'TCEQ …"
"""Featherwood Energies Pecos Pla…","""REEVES""",931.0,"""['SGT-800']""","""['Siemens Energy']""","""Permitted""","""['TCEQ NSR Air Permit', 'TCEQ …"
"""Rock House Draw Generating Sta…","""PECOS""",576.0,"""['TM2500 Gen 8', 'TM2500+']""","""['General Electric', 'General …","""Permitted""","""['TCEQ NSR Air Permit', 'TCEQ …"
"""Xenergy Monahans 1""","""WARD""",118.98,"""['HSI6CP350NG', '16M33D1280NG1…","""['Baudouin', 'Baudouin']""","""Permitted""","""['TCEQ NSR Air Permit', 'TCEQ …"
"""Poolside Data Center Pecos Cou…","""PECOS""",332.28,"""['TM2500 Gen 8', 'TM2500+']""","""['General Electric', 'General …","""Permitted""","""['TCEQ NSR Air Permit', 'TCEQ …"
"""Pecos Power Generation""","""REEVES""",478.0,"""['SGT-800']""","""['Siemens Energy']""","""Permitted""","""['TCEQ NSR Air Permit', 'TCEQ …"
"""Kilby Power Plant""","""REEVES""",2869.0,"""['7HA.02', 'Titan 350', 'Titan…","""['General Electric', 'Solar Tu…","""Permitted""","""['TCEQ NSR Air Permit', 'TCEQ …"
"""GW Ranch Energy Center""","""PECOS""",5000.0,"""['SGT-800']""","""['Siemens Energy']""","""Announced""","""['TCEQ NSR Air Permit', 'TCEQ …"
